# Langchain Messages

This notebook introduces the message-based API in LangChain, which is used to communicate with chat models. Instead of sending only a plain string, we send a structured list of messages with defined roles. This helps the model understand context, instructions, and the flow of a conversation more clearly.

A message is essentially a unit of communication between the user, the assistant, and sometimes tools. In chat-based systems, messages allow the model to remember previous context and respond in a more natural and useful way.

In [1]:
import torch
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"config\.env")

# --- ROCm/CUDA device check ---
# ROCm exposes itself to PyTorch through the same torch.cuda API as NVIDIA CUDA,
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU detected: {device_name} ({total_vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected by torch — falling back to CPU. Check your ROCm/torch install.")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

GRmodel = init_chat_model("groq:qwen/qwen3.6-27b",)

GPU detected: AMD Radeon RX 7900 XT (20.0 GB VRAM)


In [2]:
GRmodel.invoke("Please tell what is artificial intelligence?")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query**: The user asks "Please tell what is artificial intelligence?" This is a straightforward, foundational question about AI.\n\n2.  **Identify Key Components of AI Definition**:\n   - What is it? (Definition)\n   - Core capabilities/features\n   - How it works (briefly)\n   - Types/categories\n   - Real-world applications\n   - Current state/limitations\n   - Keep it clear, concise, and accessible\n\n3.  **Draft - Mental Refinement**:\n   Artificial Intelligence (AI) refers to computer systems or machines designed to perform tasks that typically require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, recognizing patterns, and making decisions. \n\n   AI works by using algorithms and large amounts of data to identify patterns, make predictions, or generate outputs. Modern AI, especially machine learning and deep learning, improves over time as it p

## Text Prompts

A text prompt is the simplest way to ask a model a question. You provide a single string, and the model responds based on that input.

This is useful for quick questions, example prompts, and basic interactions. For example, asking “What is LangChain?” sends a plain-language instruction to the model, which then generates a response.

Use when:
- you want a quick one-off answer
- the task is simple and does not need much conversation history
- you are testing a model or comparing different prompts

Text prompts are easy to start with, but for more advanced behavior, message-based prompts are often better because they let you provide roles such as system instructions and user questions.

In [4]:
GRmodel.invoke("What is langchain?")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query**: The user asks "What is langchain?" This is a straightforward definition/explanation question about a specific technology/framework.\n\n2.  **Identify Key Subject**: LangChain (often stylized as LangChain or langchain) is a popular open-source framework/library in the AI/LLM space.\n\n3.  **Core Concepts to Cover**:\n   - What it is (framework/library for building LLM-powered applications)\n   - Primary purpose (simplifies developing apps with large language models)\n   - Key features/components (chains, agents, memory, prompts, tools, retrievers, etc.)\n   - How it works/abstracts complexity (connects LLMs to data, external tools, and workflows)\n   - Ecosystem (LangChain.js, LangChain Python, LangGraph, LangSmith, etc.)\n   - Use cases (chatbots, RAG systems, data analysis, automation, etc.)\n   - Current status/context (open-source, widely adopted, maintained by LangChain AI, evolved from a li

## Message Prompts

Message prompts are more structured than plain text prompts. Instead of sending one string, we send a list of role-based messages. This gives the model clearer instructions and a better conversation context.

Types:
- System: instructions that define how the model should behave or what style it should follow.
- Human: the user's input or question.
- AI: previous assistant responses, useful when continuing a conversation.
- Tool: messages generated by tools or external systems when the model interacts with them.

### System Message
A system message sets the overall behavior of the model. It is usually used to tell the assistant how to respond, what tone to use, or what constraints to follow.

### Human Message
A human message represents the user’s actual request. This is the normal question or instruction the model should respond to.

### AI Message
An AI message represents a prior response from the assistant. It is useful when continuing a conversation and maintaining context across multiple turns.

### Tool Message
A tool message contains output from an external tool or function. It helps the model understand results from tools like search, calculators, or APIs when used in a workflow.

In [6]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence")
]

response = GRmodel.invoke(messages)
response.content

'\n<think>\nHere\'s a thinking thinking sequence\n\n1.  **Deconstruct and Analyze the Request:**\n    *   Topic: Artificial Intelligence (AI).\n    *   Goal: Write a poem.\n    *   Implicit/Explicit needs: The user wants a creative piece exploring AI. As a "poetry expert," the output should be high-quality, evocative, well-structured, and perhaps touch upon multiple facets of AI (not just one angle).\n\n2.  **Brainstorming Themes and Imagery:**\n    *   *What is AI?* Code, data, neural networks, silicon, electricity, mimicry, learning, prediction, lack of soul/consciousness, tool, mirror, future, companion.\n    *   *Metaphors:* A loom weaving words, a mirror reflecting humanity, a child learning, a god made of math, a ghost in the machine, a library that reads itself.\n    *   *Contrasts:* Warmth vs. Cold, Flesh vs. Silicon, Chaos vs. Order, Dream vs. Calculation, Creator vs. Creation.\n\n3.  **Determining Structure and Tone:**\n    *   *Structure:* Free verse might feel too modern/te